In [11]:
# Install dependencies
%pip install anthropic python-dotenv

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Create an API client
from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5"

# Helper functions to manage messages
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)
    
def chat(messages, system_prompt=None, temperature=0.7, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system_prompt:
        params["system"] = system_prompt
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# CREATE AN EVALUATION DATASET

import json 

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
print(dataset)

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

[{'task': "Write a Python function that parses an AWS S3 bucket name from an S3 URI string like 's3://my-bucket/path/to/file.txt' and returns just the bucket name."}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'company-data'."}, {'task': "Write a regex pattern that matches valid AWS EC2 instance IDs, which follow the format 'i-' followed by 8 or 17 hexadecimal characters."}]


In [13]:
# EVAL PIPELINE - Except grading

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }
    
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

[
  {
    "output": "# Python Function to Parse S3 Bucket Name\n\nHere's a clean solution with multiple approaches:\n\n## Solution 1: Simple and Readable (Recommended)\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse the bucket name from an S3 URI.\n    \n    Args:\n        s3_uri: S3 URI string (e.g., 's3://my-bucket/path/to/file.txt')\n    \n    Returns:\n        The bucket name\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI: {s3_uri}. Must start with 's3://'\")\n    \n    # Remove 's3://' prefix\n    remainder = s3_uri[5:]\n    \n    # Split by '/' to get bucket name\n    bucket_name = remainder.split('/')[0]\n    \n    if not bucket_name:\n        raise ValueError(f\"Invalid S3 URI: {s3_uri}. No bucket name found\")\n    \n    return bucket_name\n```\n\n## Solution 2: Using Regular Expressions\n\n```python\nimport re\n\ndef parse_s3

In [14]:
# EVAL PIPELINE - With model grading

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Original Task: 
    <task>{test_case['task']}</task>
    
    Solution to Evaluate: 
    <solution>{output}</solution>
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }
    
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

Average score: 7.166666666666667
[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses the bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse the bucket name from an S3 URI string.\n    \n    Args:\n        s3_uri: S3 URI string in the format 's3://bucket-name/path/to/file'\n        \n    Returns:\n        The bucket name extracted from the S3 URI\n        \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n        \n    Examples:\n        >>> parse_s3_bucket_name('s3://my-bucket/path/to/file.txt')\n        'my-bucket'\n        >>> parse_s3_bucket_name('s3://another-bucket/')\n        'another-bucket'\n        >>> parse_s3_bucket_name('s3://bucket')\n        'bucket'\n    \"\"\"\n    # Validate and strip the s3:// prefix\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI: {s3_uri}. Must start with 's3://'\")\n    \n    # Remove the 's3://' pre

In [15]:
# UPDATE DATASET FOR CODE GRADING WITH FORMAT
    
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" | "json" | "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset_with_format = generate_dataset()
print(dataset_with_format)

with open('dataset_with_format.json', 'w') as f:
    json.dump(dataset_with_format, f, indent=2)

[{'task': "Parse an AWS S3 bucket name from an S3 URI in the format 's3://bucket-name/key/path' and extract just the bucket name", 'format': 'regex'}, {'task': "Create a JSON CloudFormation template that defines a basic AWS Lambda function with a simple inline code that returns 'Hello from Lambda'", 'format': 'json'}, {'task': 'Write a Python function that takes an AWS IAM policy document (as a dictionary) and returns a list of all the actions granted in the policy', 'format': 'python'}]


In [16]:
# EVAL PIPELINE - With code grading
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not include any explanations, comments or additional text.
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

def grade_by_syntax(test_case, output):
    """Grades the output based on the expected format"""
    expected_format = test_case["format"]
    
    if expected_format == "json":
        return validate_json(output)
    elif expected_format == "python":
        return validate_python(output)
    elif expected_format == "regex":
        return validate_regex(output)
    else:
        raise ValueError(f"Unknown format: {expected_format}")

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_by_syntax(test_case, output)
    score = (model_score + syntax_score) / 2
    
    return {
        "output": output, 
        "test_case": test_case, 
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": reasoning
    }
    
with open("dataset_with_format.json", "r") as f:
    dataset_with_format = json.load(f)

results = run_eval(dataset_with_format)

print(json.dumps(results, indent=2))

Average score: 8.583333333333334
[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket(uri):\n    match = re.match(r's3://([^/]+)', uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from an S3 URI in the format 's3://bucket-name/key/path' and extract just the bucket name",
      "format": "regex"
    },
    "model_score": 7.5,
    "syntax_score": 10,
    "score": 8.75,
    "reasoning": "The solution correctly solves the core task with a well-constructed regex pattern. The regex `r's3://([^/]+)'` properly captures bucket names by matching 's3://' literally and then capturing one or more non-slash characters. The None return for non-matches is good defensive programming. However, the implementation lacks robustness in input validation and documentation. A production-ready version should validate inputs and include docstrings. The solution would benefit from handling edge cases like None input or non-string types."
  }